In [2]:
import pandas as pd
import altair as alt

df = pd.read_csv('videogamesales.csv')
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df['Year'] = df['Year'].astype(int)
df = df[(df['Year'] >= 1980) & (df['Year'] <= 2016)]

yearly_sales = (
    df.groupby('Year')['Global_Sales']
    .sum()
    .reset_index()
    .rename(columns={'Global_Sales': 'Total_Global_Sales'}))

brush = alt.selection_interval(encodings=['x'])

hover = alt.selection_point(on='mouseover', nearest=True, fields=['Year'])

line = (
    alt.Chart(yearly_sales)
    .mark_line(point=True, color='#4C78A8', strokeWidth=2.5)
    .encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='Total Global Sales (millions)', scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter(brush)
    .properties(width=700, height=300, title='Global Video Game Sales Over the Years'))

# Vertical rule that follows your mouse
rule = (
    alt.Chart(yearly_sales)
    .mark_rule(color='gray', strokeDash=[4, 4])
    .encode(
        x='Year:O',
        opacity=alt.condition(hover, alt.value(0.8), alt.value(0)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter(brush)
    .add_params(hover))

overview = (
    alt.Chart(yearly_sales)
    .mark_area(color='#4C78A8', opacity=0.3)
    .encode(
        x=alt.X('Year:O', title='', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='', axis=alt.Axis(labels=False))
    )
    .add_params(brush)
    .properties(width=700, height=80, title='Drag to select a year range'))

alt.vconcat(line + rule, overview).configure_view(strokeWidth=0)

alt.VConcatChart(...)